# 02 — Joins and GROUP BY

Purpose:
Practice joining telemetry fact rows to service and host lookup tables, then aggregating capacity metrics by service, host, environment, day, and hour.

This notebook covers:
- PostgreSQL connection
- smoke test
- run_sql helper
- table inspection helper
- INNER JOIN
- GROUP BY
- COUNT DISTINCT
- AVG / ROUND
- P95 latency with PERCENTILE_CONT
- DATE_TRUNC daily and hourly rollups
- interview explanations


## Setup 1 — Install/import dependencies


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Imports loaded.")


## Setup 2 — Connection settings


In [ ]:
DB_HOST = "host.docker.internal"
DB_PORT = 5432
DB_NAME = "studybook"
DB_USER = "sb_user"
DB_PASSWORD = "sb_pass_123"

# If this notebook runs directly on Windows instead of inside a container,
# change DB_HOST to "localhost".
password_encoded = quote_plus(DB_PASSWORD)

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{password_encoded}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

print("Database URL created.")


## Setup 3 — Smoke test connection


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user, now();"))
    row = result.fetchone()

row


## Setup 4 — Helper function to run SQL


In [ ]:
def run_sql(sql: str) -> pd.DataFrame:
    """
    Run SQL against the local PostgreSQL telemetry lab
    and return the result as a pandas DataFrame.
    """
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn)


## Setup 5 — Helper function to inspect one table safely


In [ ]:
def inspect_table_safe(table_name: str, sample_limit: int = 10):
    """
    Safe table inspector using SELECT-only queries.
    Returns a dictionary with columns, row_count, and sample_rows.
    """
    cols_sql = """
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'public' AND table_name = :table_name
    ORDER BY ordinal_position;
    """

    count_sql = f"SELECT COUNT(*) AS row_count FROM {table_name};"
    sample_sql = f"SELECT * FROM {table_name} LIMIT {sample_limit};"

    with engine.connect() as conn:
        cols = pd.read_sql_query(text(cols_sql), conn, params={"table_name": table_name})
        row_count = pd.read_sql_query(text(count_sql), conn)
        sample = pd.read_sql_query(text(sample_sql), conn)

    return {
        "columns": cols,
        "row_count": row_count,
        "sample_rows": sample,
    }


# 02 — Joins and GROUP BY Practice


## 02.1 Verify tables used in this notebook

This notebook uses:
- `telemetry_samples` = metric fact table
- `services` = service lookup table
- `hosts` = host lookup table


In [ ]:
sql = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
  AND table_name IN ('telemetry_samples', 'services', 'hosts')
ORDER BY table_name;
"""

run_sql(sql)


## 02.2 Inspect the telemetry_samples table

`telemetry_samples` contains sampled capacity metrics such as CPU, memory,
latency, request rate, error rate, and JSONB tags.


In [ ]:
inspect_table_safe("telemetry_samples")


## 02.3 Preview services

`services` turns `service_id` into readable `service_name`.


In [ ]:
sql = """
SELECT service_id, service_name
FROM services
ORDER BY service_id;
"""

run_sql(sql)


## 02.4 Preview hosts

`hosts` turns `host_id` into readable host/server metadata.


In [ ]:
sql = """
SELECT
    host_id,
    hostname,
    cluster_name,
    namespace_name,
    cloud_provider,
    instance_type,
    vcpu_allocated,
    memory_gb_allocated,
    active
FROM hosts
ORDER BY host_id;
"""

run_sql(sql)


## 02.5 INNER JOIN telemetry to services

`JOIN` means `INNER JOIN` by default.
This keeps only telemetry rows that have a matching `service_id` in `services`.


In [ ]:
sql = """
SELECT
    t.sampled_at,
    t.service_id,
    s.service_name,
    t.cpu_utilization_pct,
    t.memory_utilization_pct,
    t.p95_latency_ms,
    t.requests_per_min,
    t.error_rate_pct
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
ORDER BY
    t.sampled_at,
    s.service_name
LIMIT 20;
"""

run_sql(sql)


## 02.6 INNER JOIN telemetry to services and hosts

This joins the telemetry fact table to both lookup tables so raw IDs become
readable service and host information.


In [ ]:
sql = """
SELECT
    t.sampled_at,
    s.service_name,
    h.host_id,
    h.hostname,
    t.cpu_utilization_pct,
    t.memory_utilization_pct,
    t.p95_latency_ms,
    t.requests_per_min,
    t.error_rate_pct
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
JOIN hosts h
    ON h.host_id = t.host_id
ORDER BY
    t.sampled_at,
    s.service_name,
    h.hostname
LIMIT 20;
"""

run_sql(sql)


## 02.7 Average CPU, memory, and error rate by service

This is a JOIN + GROUP BY aggregation.
We group many telemetry rows into one summary row per service.


In [ ]:
sql = """
SELECT
    s.service_name,
    COUNT(DISTINCT t.host_id) AS host_count,
    ROUND(AVG(t.cpu_utilization_pct), 2) AS avg_cpu,
    ROUND(AVG(t.memory_utilization_pct), 2) AS avg_mem,
    ROUND(AVG(t.error_rate_pct), 2) AS avg_error_rate
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY s.service_name
ORDER BY avg_cpu DESC;
"""

run_sql(sql)


## 02.8 P95 of P95 latency by service

`p95_latency_ms` is already a sampled P95 metric.
This query calculates P95 across those sampled P95 values per service.
Call this "P95 of sampled P95 latency", not true raw request-level P95.


In [ ]:
sql = """
SELECT
    s.service_name,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY t.p95_latency_ms) AS p95_of_p95_latency
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY s.service_name
ORDER BY p95_of_p95_latency DESC;
"""

run_sql(sql)


## 02.9 Daily service rollup with DATE_TRUNC

`DATE_TRUNC('day', sampled_at)` buckets raw timestamps into daily reporting windows.
GROUP BY uses the DATE_TRUNC expression because it defines the aggregation grain.


In [ ]:
sql = """
SELECT
    DATE_TRUNC('day', t.sampled_at) AS sample_day,
    s.service_name,
    ROUND(AVG(t.cpu_utilization_pct), 2) AS avg_cpu,
    ROUND(AVG(t.requests_per_min), 0) AS avg_rpm
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY
    DATE_TRUNC('day', t.sampled_at),
    s.service_name
ORDER BY
    sample_day,
    s.service_name;
"""

run_sql(sql)


## 02.10 Hourly service rollup with DATE_TRUNC

Hourly buckets are useful when raw telemetry is collected every few minutes and we want operational trends.


In [ ]:
sql = """
SELECT
    DATE_TRUNC('hour', t.sampled_at) AS hour_bucket,
    s.service_name,
    ROUND(AVG(t.cpu_utilization_pct), 2) AS avg_cpu
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY
    DATE_TRUNC('hour', t.sampled_at),
    s.service_name
ORDER BY
    hour_bucket,
    s.service_name
LIMIT 40;
"""

run_sql(sql)


## 02.11 GROUP BY service and host

This shows whether one host inside a service is hotter than the others.


In [ ]:
sql = """
SELECT
    s.service_name,
    h.hostname,
    ROUND(AVG(t.cpu_utilization_pct), 2) AS avg_cpu,
    ROUND(MAX(t.cpu_utilization_pct), 2) AS max_cpu,
    ROUND(AVG(t.memory_utilization_pct), 2) AS avg_memory,
    ROUND(MAX(t.memory_utilization_pct), 2) AS max_memory
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
JOIN hosts h
    ON h.host_id = t.host_id
GROUP BY
    s.service_name,
    h.hostname
ORDER BY
    avg_cpu DESC,
    s.service_name,
    h.hostname;
"""

run_sql(sql)


## 02.12 GROUP BY service and environment

This query is skipped because `hosts` does not currently have an `environment` column in this lab schema.
If `hosts.environment` is added later, group by `s.service_name` and `h.environment` at that point.


## 02.13 Interview explanation

This notebook practices one of the most common telemetry SQL patterns. The telemetry table stores measurements such as CPU, memory, latency, requests, and errors. The services and hosts tables make those measurements readable. I use INNER JOIN to connect the fact table to lookup tables, then GROUP BY to summarize many raw samples into service-level or host-level capacity views. DATE_TRUNC lets me roll timestamped telemetry into daily or hourly buckets. This is useful for capacity planning because it turns noisy raw metrics into operational trends.
